# VibeScent Week 3 — 4-Signal Fusion & Reranker Ship Gate

**Author:** Harsh Agarwal  
**Pipeline version:** w3v1  
**Branch:** Text_Processing  

## Notebook Overview

This notebook implements the Week 3 pipeline on Colab.  It is designed to run
end-to-end on an A100 (preferred), L4, or T4 instance with Google Drive mounted.

### Stages
1. **Setup** — repo clone, pip install, Drive mount, secrets, GPU detection  
2. **Load Week 2 Artifacts** — embeddings + benchmark labels + enriched DataFrame  
3. **Enrichment A/B Gate** *(Decision V)* — 20 stratified rows, Gemini vs Qwen  
4. **Load Neil's CNN-CLIP Hybrid** — git fetch + checkpoint load  
5. **4-Signal Fusion Baseline** — spec weights 0.30/0.25/0.30/0.15  
6. **Grid Search Ablation** — sweep fusion weights on 20 benchmark cases  
7. **Load Reranker** — Qwen3-VL-32B-Instruct (local A100) or Gemini API fallback  
8. **Reranker Eval** — rerank top-10 per benchmark case  
9. **Metric Suite** — attribute_match@5/10, nDCG@10, delta vs baseline  
10. **Pre-registered Threshold Check** *(Decision S)* — ship / no-ship  
11. **Report Writer** — save `results/week3_report.md`  
12. **Artifact Sink** — save to Drive + local `artifacts/week3/` + git-add snippet  

### Pre-registered ship criteria (Decision S)
All three must hold to advance the reranker to Week 4:
- `attribute_match@10` delta ≥ +3 pp vs baseline
- Wins ≥ 15/20 benchmark cases
- `nDCG@10` ≥ baseline

In [ ]:
# Stage 1: Setup
# uv for 50-100x faster installs, depth-1 clone, parallel Drive+secrets, tqdm ETAs
import os, subprocess, sys, time, threading
_t0 = time.perf_counter()

REPO_URL  = "https://github.com/Harsh-Agarwal0/vibescent.git"
REPO_DIR  = "/content/vibescent"
PIPELINE_VERSION = "w3v1"

# ── Bootstrap uv (Rust package manager, 50-100x faster than pip) ──────────────
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "uv"],
               check=True, capture_output=True)
print("uv ready")

# ── Shallow clone / fast-forward pull ────────────────────────────────────────
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth=1", "--single-branch", "--branch=Text_Processing",
         REPO_URL, REPO_DIR],
        check=True, capture_output=True,
    )
    print("Repo cloned (shallow)")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
                   capture_output=True)
    print("Repo up to date")
os.chdir(REPO_DIR)

# ── Parallel: Drive mount + secrets + pkg install ────────────────────────────
from tqdm.auto import tqdm
steps = ["Mount Drive", "Load secrets", "Install pkg", "Install GPU deps", "GPU detect"]
bar = tqdm(steps, desc="Stage 1 -- Setup", ncols=100, unit="step")

# Thread 1: Mount Drive
_drive_exc = None
def _mount_drive():
    global _drive_exc, DRIVE_BASE, W2_ARTIFACTS, W3_ARTIFACTS
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_BASE    = "/content/drive/MyDrive/vibescent"
        W2_ARTIFACTS  = DRIVE_BASE          # Week 2 saves directly to DRIVE_BASE
        W3_ARTIFACTS  = f"{DRIVE_BASE}/week3"
        os.makedirs(W3_ARTIFACTS, exist_ok=True)
    except Exception as e:
        _drive_exc = e
        DRIVE_BASE   = "artifacts"
        W2_ARTIFACTS = "artifacts"
        W3_ARTIFACTS = "artifacts/week3"
        os.makedirs(W3_ARTIFACTS, exist_ok=True)

_drive_thread = threading.Thread(target=_mount_drive, daemon=False)
_drive_thread.start()

# Thread 2: Load secrets
_gemini_key = None
_voyage_key = None
def _load_secrets():
    global _gemini_key, _voyage_key
    try:
        from google.colab import userdata
        _gemini_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        _gemini_key = os.environ.get("GEMINI_API_KEY", "")
    try:
        from google.colab import userdata
        _voyage_key = userdata.get("VOYAGE_API_KEY")
    except Exception:
        _voyage_key = os.environ.get("VOYAGE_API_KEY", "")

_sec_thread = threading.Thread(target=_load_secrets, daemon=False)
_sec_thread.start()

bar.update(1)  # "Mount Drive" (started)

_drive_thread.join(); _sec_thread.join()
bar.update(1)  # "Load secrets"

if _drive_exc:
    print(f"\n[WARNING] Drive mount failed ({_drive_exc}). "
          "Using local artifacts/ fallback. W2 embeddings must exist locally.")
else:
    print(f"\n  Drive OK -> {DRIVE_BASE}")

os.environ["GEMINI_API_KEY"]  = _gemini_key or ""
os.environ["GOOGLE_API_KEY"]  = _gemini_key or ""
if not _gemini_key:
    print("\n  [WARNING] GEMINI_API_KEY not found in Colab Secrets.")
    print("  Add it via the key icon in the left sidebar.")

# ── Install vibescents package (uv) ──────────────────────────────────────────
bar.set_postfix_str("uv install vibescents...")
subprocess.run(
    ["uv", "pip", "install", "--system", "-q", "-e", "."],
    check=True, capture_output=True,
)
bar.update(1)  # "Install pkg"

# ── Install GPU/runtime deps (uv, one shot) ───────────────────────────────────
bar.set_postfix_str("uv install GPU deps (~2 min first run)...")
gpu_deps = [
    "google-genai", "json-repair", "tqdm",
    "sentence-transformers>=3.4,<4.0",
    "accelerate>=1.3,<2.0",
    "bitsandbytes>=0.45,<0.46",
    "transformers>=4.57.0",
    "vllm>=0.8,<0.9",
    "outlines>=0.1,<0.2",
    "torchvision",
    "pillow",
]
subprocess.run(
    ["uv", "pip", "install", "--system", "-q"] + gpu_deps,
    check=True, capture_output=True,
)
bar.update(1)  # "Install GPU deps"

# ── GPU detection ─────────────────────────────────────────────────────────────
import torch
from vibescents.week2_pipeline import detect_gpu_tier, check_disk_space

try:
    GPU_TIER = detect_gpu_tier()
except RuntimeError:
    GPU_TIER = "CPU"

USE_LOCAL_LLM = GPU_TIER == "A100"
USE_LOCAL_MM  = GPU_TIER in ("A100", "L4")
bar.update(1)  # "GPU detect"
bar.close()

if os.path.exists("/content"):
    check_disk_space("/content")

os.makedirs("artifacts/week3", exist_ok=True)
os.makedirs("results", exist_ok=True)

print(f"\nGPU: {GPU_TIER} | LOCAL_LLM={USE_LOCAL_LLM} | LOCAL_MM={USE_LOCAL_MM}")
print(f"Drive:        {DRIVE_BASE}")
print(f"W2 artifacts: {W2_ARTIFACTS}")
print(f"W3 artifacts: {W3_ARTIFACTS}")
print(f"Setup done in {time.perf_counter()-_t0:.1f}s")


In [ ]:
# Stage 2: Load Week 2 Artifacts
# Parallel .npy loads via ThreadPoolExecutor, sanity checks, tqdm
PIPELINE_VERSION = globals().get("PIPELINE_VERSION", "w3v1")
import json, time, os
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from vibescents.week2_pipeline import embedding_sanity_check, read_manifest
from vibescents.schemas import BenchmarkCaseLabel
_t0 = time.perf_counter()

def _require(path, label):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"[REQUIRED] {label} not found at: {path}\n"
            "  -> Run Week 2 P0+P1 on Colab first and ensure Drive is mounted correctly."
        )
    return path

ARTIFACT_MAP = {
    "text_full":    f"{W2_ARTIFACTS}/fragrance_raw_full/embeddings.npy",
    "enriched_2k":  f"{W2_ARTIFACTS}/fragrance_enriched_2k/embeddings.npy",
    "enriched_500": f"{W2_ARTIFACTS}/fragrance_enriched_500/embeddings.npy",
    "mm_2k":        f"{W2_ARTIFACTS}/multimodal_2k/doc_embeddings.npy",
    "occasions":    f"{W2_ARTIFACTS}/occasions/embeddings.npy",
}
BENCHMARK_PATH  = f"{W2_ARTIFACTS}/benchmark_cases.json"
ENRICHED_DF_PATH = f"{W2_ARTIFACTS}/vibescent_enriched_2k_v2.csv"

# Check all required paths before loading anything
missing = []
for label, path in list(ARTIFACT_MAP.items()) + [("benchmark", BENCHMARK_PATH), ("csv", ENRICHED_DF_PATH)]:
    if not os.path.exists(path):
        missing.append(f"  {label}: {path}")
if missing:
    print("[ERROR] Missing Week 2 artifacts:")
    for m in missing:
        print(m)
    print("\nSolution: Run Week 2 notebook (P0+P1) first, then re-run this cell.")
    raise FileNotFoundError("Week 2 artifacts missing - see above")

print("All Week 2 paths verified. Loading in parallel...")

def _load_npy(key_path):
    key, path = key_path
    arr = np.load(path)
    return key, arr

embeddings = {}
pbar = tqdm(ARTIFACT_MAP.items(), total=len(ARTIFACT_MAP), desc="Loading embeddings", ncols=90)
with ThreadPoolExecutor(max_workers=5) as pool:
    futures = {pool.submit(_load_npy, (k, v)): k for k, v in ARTIFACT_MAP.items()}
    for fut in as_completed(futures):
        k, arr = fut.result()
        embeddings[k] = arr
        pbar.set_postfix_str(f"{k} {arr.shape}")
        pbar.update(1)
pbar.close()

emb_text_full   = embeddings["text_full"]
emb_enriched_2k = embeddings["enriched_2k"]
emb_mm_2k       = embeddings["mm_2k"]
emb_occasions   = embeddings["occasions"]

# Quick sanity check
for name, arr in [("enriched_2k", emb_enriched_2k), ("mm_2k", emb_mm_2k)]:
    embedding_sanity_check(arr)
print("Embedding sanity checks passed")

# Benchmark cases
with open(BENCHMARK_PATH, encoding="utf-8") as fh:
    _raw = json.load(fh)
benchmark_cases = [BenchmarkCaseLabel.model_validate(c) for c in _raw]
if len(benchmark_cases) != 20:
    print(f"[WARNING] Expected 20 benchmark cases, got {len(benchmark_cases)}")
print(f"Benchmark cases: {len(benchmark_cases)}")

# Enriched DataFrame
print("Loading enriched DataFrame...", end=" ", flush=True)
df_enriched = pd.read_csv(ENRICHED_DF_PATH)
if len(df_enriched) != len(emb_enriched_2k):
    print(f"\n[WARNING] DataFrame rows ({len(df_enriched)}) != embeddings ({len(emb_enriched_2k)})")
    print("  Truncating to min length to avoid index errors.")
    min_len = min(len(df_enriched), len(emb_enriched_2k))
    df_enriched   = df_enriched.iloc[:min_len].reset_index(drop=True)
    emb_enriched_2k = emb_enriched_2k[:min_len]
    emb_mm_2k       = emb_mm_2k[:min_len]
if "fragrance_id" not in df_enriched.columns:
    df_enriched.insert(0, "fragrance_id", df_enriched.index.astype(str))
print(f"done. shape={df_enriched.shape}")

print(f"\nAll Week 2 artifacts loaded in {time.perf_counter()-_t0:.1f}s")
print("Shapes:")
for k, arr in embeddings.items():
    print(f"  {k}: {arr.shape}")


In [ ]:
# Stage 3: Enrichment A/B Gate (Decision V)
# Stratified 20-row comparison: Gemini vs Qwen3.5-27B enrichment quality
PIPELINE_VERSION = globals().get("PIPELINE_VERSION", "w3v1")
import time, os
import pandas as pd
from tqdm.auto import tqdm
from vibescents.enrich import GeminiEnrichmentClient, _build_prompt as _ep
from vibescents.schemas import EnrichmentSchemaV2
_t0 = time.perf_counter()

AB_CACHE = f"{W3_ARTIFACTS}/ab_gate_comparison.csv"

if os.path.exists(AB_CACHE):
    ab_comparison = pd.read_csv(AB_CACHE)
    gemini_mean = float(ab_comparison["gemini_score"].mean())
    qwen_mean   = float(ab_comparison["qwen_score"].mean())
    print(f"Loaded cached A/B gate results ({len(ab_comparison)} rows)")
    print(f"  Gemini mean: {gemini_mean:.3f}  Qwen mean: {qwen_mean:.3f}")
else:
    # ── Stratified sampling ───────────────────────────────────────────────────
    STRATA = {
        "woody_oriental": ["woody", "oriental", "oud", "amber", "sandalwood", "patchouli"],
        "fresh_citrus":   ["fresh", "citrus", "lemon", "bergamot", "aquatic", "green"],
        "floral":         ["floral", "rose", "jasmine", "iris", "peony", "violet"],
        "gourmand":       ["gourmand", "vanilla", "caramel", "chocolate", "tonka", "praline"],
    }

    def _matches(row, keywords):
        text = " ".join(str(row.get(c, "") or "") for c in
                        ["main_accords", "top_notes", "middle_notes", "base_notes"]).lower()
        return any(kw in text for kw in keywords)

    selected_rows = []
    for stratum, kws in STRATA.items():
        mask = df_enriched.apply(_matches, axis=1, keywords=kws)
        pool = df_enriched[mask]
        pick = pool.head(5) if len(pool) >= 5 else pd.concat(
            [pool, df_enriched[~mask].head(5 - len(pool))], ignore_index=True)
        selected_rows.append(pick.head(5))
        print(f"  Stratum '{stratum}': {len(pick)} rows")

    ab_df = pd.concat(selected_rows, ignore_index=True).head(20)
    print(f"A/B sample: {len(ab_df)} rows")

    # ── Quality scoring heuristic ─────────────────────────────────────────────
    def _score(s):
        score  = 0.3 if len(s.vibe_sentence.split()) >= 8 else 0.0
        score += 0.2 if len(s.character_tags) >= 3 else 0.1
        score += 0.2 if len(s.mood_tags) >= 1 else 0.0
        score += 0.15 if s.likely_season in {"spring","summer","fall","winter","all-season"} else 0.0
        score += 0.15 if 0.0 <= s.formality <= 1.0 else 0.0
        return round(score, 3)

    # ── Gemini run ────────────────────────────────────────────────────────────
    print("\nRunning Gemini enrichment on 20 rows...")
    gemini_client = GeminiEnrichmentClient()
    gemini_scores, gemini_vibes = [], []
    import time as _t
    for _, row in tqdm(ab_df.iterrows(), total=len(ab_df), desc="Gemini A/B", ncols=90):
        try:
            result = gemini_client.generate(_ep(row))
            score  = _score(result)
            gemini_scores.append(score)
            gemini_vibes.append(result.vibe_sentence[:80])
        except Exception as e:
            gemini_scores.append(0.0)
            gemini_vibes.append(f"ERROR: {e}"[:80])
        _t.sleep(0.3)  # rate-limit guard

    gemini_mean = sum(gemini_scores) / len(gemini_scores)
    print(f"Gemini mean: {gemini_mean:.3f}")

    # ── Qwen run ──────────────────────────────────────────────────────────────
    print("\nRunning Qwen enrichment on 20 rows...")
    if USE_LOCAL_LLM:
        from vibescents.enrich import QwenOutlinesEnrichmentClient
        qwen_client = QwenOutlinesEnrichmentClient(
            model_name="Qwen/Qwen3.5-27B-GPTQ-Int4"
        )
    else:
        qwen_client = gemini_client  # fallback: both Gemini (no local LLM)
        print("  [INFO] No A100 detected -- Qwen A/B skipped, using Gemini for both providers")

    qwen_scores, qwen_vibes = [], []
    for _, row in tqdm(ab_df.iterrows(), total=len(ab_df), desc="Qwen A/B", ncols=90):
        try:
            result = qwen_client.generate(_ep(row))
            score  = _score(result)
            qwen_scores.append(score)
            qwen_vibes.append(result.vibe_sentence[:80])
        except Exception as e:
            qwen_scores.append(0.0)
            qwen_vibes.append(f"ERROR: {e}"[:80])

    qwen_mean = sum(qwen_scores) / len(qwen_scores)
    print(f"Qwen mean: {qwen_mean:.3f}")

    ab_comparison = pd.DataFrame({
        "fragrance_id":  ab_df["fragrance_id"].values,
        "name":          ab_df.get("name", pd.Series(["?"]*len(ab_df))).values,
        "gemini_score":  gemini_scores,
        "qwen_score":    qwen_scores,
        "gemini_vibe":   gemini_vibes,
        "qwen_vibe":     qwen_vibes,
    })
    ab_comparison.to_csv(AB_CACHE, index=False)
    print(f"Saved A/B comparison to {AB_CACHE}")

# ── Decision gate ─────────────────────────────────────────────────────────────
qwen_delta    = qwen_mean - gemini_mean
ENRICH_PROVIDER = "qwen" if (qwen_mean >= gemini_mean - 0.5 and USE_LOCAL_LLM) else "gemini"
gate_pass       = qwen_mean >= gemini_mean - 0.5

print("\n" + "=" * 55)
print(f"A/B gate: Gemini={gemini_mean:.3f}  Qwen={qwen_mean:.3f}  delta={qwen_delta:+.3f}")
print(f"Threshold: Qwen >= Gemini - 0.5  ->  {'PASS' if gate_pass else 'FAIL'}")
print(f"Provider for downstream enrichment: {ENRICH_PROVIDER.upper()}")
print("=" * 55)
if not gate_pass:
    print("[WARNING] Qwen quality is notably below Gemini.")
    print("  Proceeding with Gemini for Week 3 enrichment.")
print(f"\nStage 3 done in {time.perf_counter()-_t0:.0f}s")


In [ ]:
# Stage 4: Load Neil's CNN-CLIP Hybrid
# Critical: checkout models/cnn_clip_hybrid.py from origin/Image_Processing
# Checkpoint: checkpoints/hybrid/best.pt (Neil updated to hybrid in latest commit)
PIPELINE_VERSION = globals().get("PIPELINE_VERSION", "w3v1")
import os, sys, time, subprocess, torch
from tqdm.auto import tqdm
from pathlib import Path
from vibescents.image_scoring import NeilCNNWrapper
from vibescents.image_preprocess import decode_b64_to_cnn_tensor
_t0 = time.perf_counter()

# ── Fetch Neil's latest model files from Image_Processing branch ──────────────
steps = ["git fetch Image_Processing", "checkout hybrid model", "checkout checkpoint",
         "load checkpoint", "verify forward pass"]
bar = tqdm(steps, desc="Stage 4 -- Load CNN", ncols=100, unit="step")

bar.set_postfix_str("fetching...")
subprocess.run(
    ["git", "-C", REPO_DIR, "fetch", "origin", "Image_Processing"],
    check=True, capture_output=True,
)
bar.update(1)

# Checkout Neil's model files directly into our working directory
for git_path in [
    "models/cnn_clip_hybrid.py",
    "models/cnn_baseline.py",
    "models/clip_standalone.py",
]:
    subprocess.run(
        ["git", "-C", REPO_DIR, "checkout", "origin/Image_Processing", "--", git_path],
        capture_output=True,  # non-fatal: file may not exist in all branches
    )
models_dir = Path(REPO_DIR) / "models"
if str(models_dir) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
bar.update(1)

# Checkout checkpoint: prefer hybrid (latest/best), fallback to cnn
CKPT_PATH = None
for ckpt_candidate in ["checkpoints/hybrid/best.pt", "checkpoints/cnn/best.pt"]:
    result = subprocess.run(
        ["git", "-C", REPO_DIR, "checkout", "origin/Image_Processing", "--", ckpt_candidate],
        capture_output=True,
    )
    if result.returncode == 0 and Path(f"{REPO_DIR}/{ckpt_candidate}").exists():
        CKPT_PATH = f"{REPO_DIR}/{ckpt_candidate}"
        print(f"\n  Checkpoint: {ckpt_candidate}")
        break

if CKPT_PATH is None:
    print("\n[WARNING] Could not fetch checkpoint from Image_Processing branch.")
    print("  Make sure 'git fetch origin Image_Processing' succeeded and Neil's checkpoint is committed.")
    raise FileNotFoundError(
        "Neil's CNN checkpoint not found. "
        "Verify checkpoints/hybrid/best.pt or checkpoints/cnn/best.pt exists on origin/Image_Processing."
    )
bar.update(1)

# ── Import CNNCLIPHybrid from Neil's checked-out file ─────────────────────────
bar.set_postfix_str("loading weights into model...")
CNN_CLASS = None

try:
    from models.cnn_clip_hybrid import CNNCLIPHybrid
    CNN_CLASS = CNNCLIPHybrid
    print("  Imported CNNCLIPHybrid from models.cnn_clip_hybrid")
except ImportError as e:
    print(f"  Import failed ({e}), trying cnn_baseline...")
    try:
        from models.cnn_baseline import CNNBaseline as CNN_CLASS
        print("  Fell back to CNNBaseline")
    except ImportError:
        CNN_CLASS = None

if CNN_CLASS is None:
    # Last-resort inline definition (matches Neil's architecture, transformers-based)
    print("  Using inline CNNCLIPHybrid compatible definition (transformers.CLIPModel)")
    import torch.nn as nn
    import torchvision.models as tvm
    from transformers import CLIPModel

    class CNNCLIPHybrid(nn.Module):
        CNN_DIM = 2048; CLIP_DIM = 768; FUSED_DIM = 512
        def __init__(self):
            super().__init__()
            resnet = tvm.resnet50(weights=None)
            self.cnn_backbone = nn.Sequential(*list(resnet.children())[:-1])
            clip = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
            self.clip_vision = clip.vision_model
            self.clip_proj   = clip.visual_projection
            for p in self.clip_vision.parameters(): p.requires_grad = False
            for p in self.clip_proj.parameters():   p.requires_grad = False
            self.fusion = nn.Sequential(
                nn.Linear(self.CNN_DIM + self.CLIP_DIM, self.FUSED_DIM),
                nn.BatchNorm1d(self.FUSED_DIM), nn.ReLU(inplace=True), nn.Dropout(0.3)
            )
            self.formal_head    = nn.Linear(self.FUSED_DIM, 3)
            self.season_head    = nn.Linear(self.FUSED_DIM, 4)
            self.gender_head    = nn.Linear(self.FUSED_DIM, 3)
            self.time_head      = nn.Linear(self.FUSED_DIM, 2)
            self.frequency_head = nn.Linear(self.FUSED_DIM, 2)

        def forward(self, x):
            cnn_f  = self.cnn_backbone(x).flatten(1)
            clip_f = self.clip_proj(self.clip_vision(pixel_values=x).pooler_output)
            fused  = self.fusion(torch.cat([cnn_f, clip_f], dim=1))
            return {
                "formal": self.formal_head(fused), "season": self.season_head(fused),
                "gender": self.gender_head(fused), "time":   self.time_head(fused),
                "frequency": self.frequency_head(fused),
            }
    CNN_CLASS = CNNCLIPHybrid
    print("  Inline fallback ready")

# ── Load checkpoint ───────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
neil_wrapper = NeilCNNWrapper.from_checkpoint(
    checkpoint_path=CKPT_PATH,
    model_builder=CNN_CLASS,
    device=device,
)
bar.update(1)

# ── Verify forward pass ───────────────────────────────────────────────────────
import torchvision.transforms.functional as TF
from PIL import Image as _PIL
_dummy = _PIL.fromarray(
    (torch.rand(224, 224, 3) * 255).byte().numpy().astype("uint8"))
_t_dummy = TF.normalize(
    TF.to_tensor(_dummy), mean=[0.48145466, 0.4578275, 0.40821073],
    std=[0.26862954, 0.26130258, 0.27577711]
).unsqueeze(0).to(device)
_probs = neil_wrapper.predict_head_probabilities(_t_dummy)
assert _probs.formal.shape == (3,), f"formal head shape mismatch: {_probs.formal.shape}"
assert _probs.season.shape == (4,), f"season head shape mismatch: {_probs.season.shape}"
assert _probs.time.shape   == (2,), f"time head shape mismatch: {_probs.time.shape}"
bar.update(1)
bar.close()

print(f"\nNeil CNN-CLIP Hybrid loaded | device={device} | {time.perf_counter()-_t0:.0f}s")
print(f"  Checkpoint: {CKPT_PATH}")
print(f"  Forward pass verified: formal={_probs.formal}, season={_probs.season}, time={_probs.time}")


In [ ]:
# Stage 5: 4-Signal Fusion Baseline (spec weights 0.30/0.25/0.30/0.15)
# Vectorised numpy scoring -- no Python row loops
PIPELINE_VERSION = globals().get("PIPELINE_VERSION", "w3v1")
import time, json, os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from vibescents.fusion import DEFAULT_FUSION_WEIGHTS, fuse_scores, normalize_signal_map
from vibescents.image_scoring import score_candidate_pool
from vibescents.similarity import cosine_similarity_matrix, top_k_indices
_t0 = time.perf_counter()

N_CORPUS = len(df_enriched)
N_CASES  = len(benchmark_cases)

# ── Pre-build binary relevance matrix (n_cases x n_corpus) ────────────────────
# Used by the vectorised grid search and by per-case metric computation.
print("Pre-computing binary relevance matrix...", end=" ", flush=True)
rel_season    = np.zeros((N_CASES, N_CORPUS), dtype=np.float32)
rel_formality = np.zeros((N_CASES, N_CORPUS), dtype=np.float32)
rel_daynnight = np.zeros((N_CASES, N_CORPUS), dtype=np.float32)

seasons_arr  = df_enriched["likely_season"].fillna("all-season").str.lower().values
formality_arr = pd.to_numeric(df_enriched.get("formality", pd.Series([0.5]*N_CORPUS)), errors="coerce").fillna(0.5).values
daynight_arr  = pd.to_numeric(df_enriched.get("day_night",  pd.Series([0.5]*N_CORPUS)), errors="coerce").fillna(0.5).values

for ci, case in enumerate(benchmark_cases):
    ts = case.target_season.lower()
    rel_season[ci]    = ((seasons_arr == ts) | (seasons_arr == "all-season")).astype(np.float32)
    tf = case.target_formality.lower()
    if tf == "casual":
        rel_formality[ci] = (formality_arr < 0.33).astype(np.float32)
    elif tf == "formal":
        rel_formality[ci] = (formality_arr >= 0.67).astype(np.float32)
    else:
        rel_formality[ci] = ((formality_arr >= 0.33) & (formality_arr < 0.67)).astype(np.float32)
    tdn = case.target_day_night.lower()
    if tdn == "day":
        rel_daynnight[ci] = (daynight_arr < 0.5).astype(np.float32)
    elif tdn == "night":
        rel_daynnight[ci] = (daynight_arr >= 0.5).astype(np.float32)
    else:
        rel_daynnight[ci] = np.ones(N_CORPUS, dtype=np.float32)

binary_relevance = (rel_season * rel_formality * rel_daynnight)  # (N_CASES, N_CORPUS)
print(f"done. Mean relevance density: {binary_relevance.mean():.4f}")

# ── Occasion-to-query mapping heuristic ──────────────────────────────────────
OCCASION_LABELS = [
    "casual everyday", "romantic evening", "office professional",
    "outdoor adventure", "formal gala", "beach summer", "winter holiday", "gym workout",
]

def _case_query_emb(case):
    text = case.occasion_text.lower()
    scores = np.array([sum(1 for w in lbl.split() if w in text) for lbl in OCCASION_LABELS])
    best = int(np.argmax(scores)) if scores.max() > 0 else 0
    return emb_occasions[best]

# ── Build per-case signal matrices ────────────────────────────────────────────
print("Building per-case signal matrices (4 signals x 20 cases x 2000 corpus)...")
agg_text   = np.zeros((N_CASES, N_CORPUS), dtype=np.float32)
agg_mm     = np.zeros((N_CASES, N_CORPUS), dtype=np.float32)
agg_img    = np.zeros((N_CASES, N_CORPUS), dtype=np.float32)
agg_struct = np.zeros((N_CASES, N_CORPUS), dtype=np.float32)

# Pre-compute structured scores (vectorised per case)
struct_scores_all = (rel_season + rel_formality + rel_daynnight)  # (N_CASES, N_CORPUS) in [0,3]

# Pre-compute image scores: reuse one neutral query image for all cases (benchmark has no real images)
_dummy_probs = neil_wrapper.predict_head_probabilities(_t_dummy)  # from Stage 4
pbar = tqdm(enumerate(benchmark_cases), total=N_CASES, desc="Building signals", ncols=100)
for ci, case in pbar:
    q_emb = _case_query_emb(case)
    agg_text[ci]   = cosine_similarity_matrix(q_emb[np.newaxis], emb_enriched_2k)[0]
    agg_mm[ci]     = cosine_similarity_matrix(q_emb[np.newaxis], emb_mm_2k)[0]
    agg_img[ci]    = score_candidate_pool(_dummy_probs, df_enriched.to_dict("records"))
    agg_struct[ci] = struct_scores_all[ci]
    pbar.set_postfix(case=case.case_id[:20])
pbar.close()

print("Signal matrices built. Running baseline fusion (spec weights)...")

# ── Baseline fusion ───────────────────────────────────────────────────────────
W = DEFAULT_FUSION_WEIGHTS
baseline_results = []
am5_list, am10_list, ndcg10_list = [], [], []

for ci, case in tqdm(enumerate(benchmark_cases), total=N_CASES, desc="Baseline fusion", ncols=90):
    score_map = {
        "text":       agg_text[ci],
        "multimodal": agg_mm[ci],
        "image":      agg_img[ci],
        "structured": agg_struct[ci],
    }
    fused    = fuse_scores(score_map, W)
    top10idx = top_k_indices(fused, 10)
    top5idx  = top10idx[:5]
    rel_v    = binary_relevance[ci]

    am5  = float(rel_v[top5idx].mean())
    am10 = float(rel_v[top10idx].mean())
    # nDCG@10 using binary relevance
    dcg  = sum(float(rel_v[top10idx[r]]) / np.log2(r+2) for r in range(10))
    idcg = sum(1.0 / np.log2(r+2) for r in range(10))
    ndcg = float(dcg / idcg) if idcg > 0 else 0.0

    am5_list.append(am5); am10_list.append(am10); ndcg10_list.append(ndcg)
    baseline_results.append({
        "case_id": case.case_id, "fused_scores": fused,
        "top10_idx": top10idx, "am@5_baseline": am5,
        "am@10_baseline": am10, "ndcg@10_baseline": ndcg,
    })

mean_baseline_am5  = float(np.mean(am5_list))
mean_baseline_am10 = float(np.mean(am10_list))
mean_baseline_ndcg = float(np.mean(ndcg10_list))

print(f"\nBaseline (spec weights):")
print(f"  mean am@5  = {mean_baseline_am5:.4f}")
print(f"  mean am@10 = {mean_baseline_am10:.4f}")
print(f"  mean nDCG@10 = {mean_baseline_ndcg:.4f}")
print(f"Stage 5 done in {time.perf_counter()-_t0:.0f}s")


In [ ]:
# Stage 6: Vectorised Grid Search (numpy einsum -- ~220 configs in milliseconds)
# No Python loop over weight combos -- fully numpy-vectorised.
PIPELINE_VERSION = globals().get("PIPELINE_VERSION", "w3v1")
import time, json, os
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path
from vibescents.fusion import DEFAULT_FUSION_WEIGHTS, build_weight_grid, normalize_signal_map
from vibescents.similarity import top_k_indices
_t0 = time.perf_counter()

GRID_CACHE = f"{W3_ARTIFACTS}/grid_search_results.json"

if os.path.exists(GRID_CACHE):
    with open(GRID_CACHE) as fh:
        _cached = json.load(fh)
    OPTIMUM_WEIGHTS = _cached["optimum_weights"]
    print(f"Loaded cached grid search: {OPTIMUM_WEIGHTS}  metric={_cached['optimum_metric']:.4f}")
else:
    GRID_STEP = 0.05

    # ── Normalise signal matrices once ────────────────────────────────────────
    print("Normalising signal matrices...", end=" ", flush=True)
    def _minmax(mat):
        mn = mat.min(axis=1, keepdims=True)
        mx = mat.max(axis=1, keepdims=True)
        denom = np.where(mx - mn < 1e-8, 1.0, mx - mn)
        return (mat - mn) / denom

    sig_t = _minmax(agg_text)    # (N_CASES, N_CORPUS)
    sig_m = _minmax(agg_mm)
    sig_i = _minmax(agg_img)
    sig_s = _minmax(agg_struct)
    signals_norm = np.stack([sig_t, sig_m, sig_i, sig_s], axis=2)  # (N_CASES, N_CORPUS, 4)
    print("done")

    # ── Weight grid ────────────────────────────────────────────────────────────
    weight_grid = build_weight_grid(
        channels=("text", "multimodal", "image", "structured"),
        step=GRID_STEP,
    )
    wg = np.array([[w["text"], w["multimodal"], w["image"], w["structured"]]
                   for w in weight_grid], dtype=np.float32)  # (N_CONFIGS, 4)
    N_CONFIGS = len(weight_grid)
    print(f"Grid: {N_CONFIGS} configs, step={GRID_STEP}")

    # ── Vectorised fused scores for ALL configs at once ───────────────────────
    # signals_norm: (N_CASES, N_CORPUS, 4) @ wg.T (4, N_CONFIGS) = (N_CASES, N_CORPUS, N_CONFIGS)
    print("Running vectorised grid search...", end=" ", flush=True)
    _gs_t = time.perf_counter()
    fused_all = signals_norm @ wg.T           # (N_CASES, N_CORPUS, N_CONFIGS)
    print(f"fused matrix built in {time.perf_counter()-_gs_t:.2f}s. shape={fused_all.shape}")

    # Top-10 indices along corpus axis for each (case, config)
    # np.argpartition: (N_CASES, N_CORPUS, N_CONFIGS) -> pick top 10 along axis=1
    # Doing per-config to keep memory under 2 GB
    K = 10
    am10_per_config = np.zeros(N_CONFIGS, dtype=np.float32)

    pbar = tqdm(range(N_CONFIGS), desc="Scoring configs", ncols=90, unit="cfg",
                mininterval=0.5)
    for j in pbar:
        fused_j    = fused_all[:, :, j]                   # (N_CASES, N_CORPUS)
        top10_idx  = np.argpartition(-fused_j, K, axis=1)[:, :K]  # (N_CASES, K)
        # gather binary_relevance
        case_idx   = np.arange(N_CASES)[:, None]
        hits       = binary_relevance[case_idx, top10_idx] # (N_CASES, K)
        am10_per_config[j] = float(hits.mean())
        if j % 20 == 0:
            pbar.set_postfix(best=f"{am10_per_config[:j+1].max():.4f}")
    pbar.close()

    best_j = int(am10_per_config.argmax())
    OPTIMUM_WEIGHTS = weight_grid[best_j]
    _opt_metric = float(am10_per_config[best_j])

    grid_result_metric = _opt_metric
    print(f"\nOptimum weights: {OPTIMUM_WEIGHTS}")
    print(f"Optimum am@10:   {_opt_metric:.4f}  (baseline: {mean_baseline_am10:.4f})")
    print(f"Grid search done in {time.perf_counter()-_gs_t:.2f}s for {N_CONFIGS} configs")

    # Save cache
    os.makedirs(os.path.dirname(GRID_CACHE), exist_ok=True)
    with open(GRID_CACHE, "w") as fh:
        json.dump({
            "optimum_weights": OPTIMUM_WEIGHTS,
            "optimum_metric":  _opt_metric,
            "spec_weights":    DEFAULT_FUSION_WEIGHTS,
            "grid_step":       GRID_STEP,
            "grid_size":       N_CONFIGS,
        }, fh, indent=2)
    print(f"Grid results saved to {GRID_CACHE}")

# ── Compare spec vs optimum ───────────────────────────────────────────────────
print("\nSpec weights vs optimum:")
for ch in ("text", "multimodal", "image", "structured"):
    spec = DEFAULT_FUSION_WEIGHTS[ch]
    opt  = OPTIMUM_WEIGHTS.get(ch, spec)
    delta = opt - spec
    print(f"  {ch:12s}  spec={spec:.2f}  opt={opt:.2f}  delta={delta:+.2f}")

print(f"\nStage 6 done in {time.perf_counter()-_t0:.0f}s")


In [ ]:
# Stage 7: Load Reranker (Qwen3-VL-32B-Instruct local or Gemini fallback)
PIPELINE_VERSION = globals().get("PIPELINE_VERSION", "w3v1")
import time
import torch
from tqdm.auto import tqdm
from vibescents.week2_pipeline import check_disk_space
_t0 = time.perf_counter()

steps = ["unload embedder if any", "load reranker"]
bar = tqdm(steps, desc="Stage 7 -- Load reranker", ncols=90, unit="step")

# Free VRAM before loading a 32B model
import gc
for _var in ["embedder", "vl_embedder"]:
    if _var in dir():
        del globals()[_var]
gc.collect()
torch.cuda.empty_cache()
bar.update(1)

RERANKER_BACKEND = None
reranker = None

if USE_LOCAL_LLM:
    bar.set_postfix_str("loading Qwen3-VL-32B-Instruct via vLLM (~18 GB AWQ)...")
    try:
        from vllm import LLM, SamplingParams
        _reranker_llm = LLM(
            model="Qwen/Qwen3-VL-32B-Instruct",
            quantization="awq",
            max_model_len=8192,
            gpu_memory_utilization=0.88,
            trust_remote_code=True,
        )
        _sampling = SamplingParams(temperature=0.0, max_tokens=2048)

        class QwenVLReranker:
            SYSTEM = (
                "You are an expert fragrance stylist. "
                "Rank candidates for the user's occasion. "
                "Return a JSON array: [{fragrance_id, score (0-1), reasoning}]. "
                "Only scores >0.7 for strong matches."
            )
            def rerank(self, *, occasion_text, candidates, image_path=None):
                lines = [f"Occasion: {occasion_text}", "", "Candidates:"]
                for c in candidates:
                    lines.append(
                        f"  id={c['fragrance_id']}  name={c.get('name','')}  "
                        f"season={c.get('likely_season','')}  "
                        f"vibe={c.get('retrieval_text','')[:120]}"
                    )
                prompt = "\n".join(lines)
                from transformers import AutoTokenizer
                tok = AutoTokenizer.from_pretrained(
                    "Qwen/Qwen3-VL-32B-Instruct", trust_remote_code=True)
                msgs = [{"role": "system", "content": self.SYSTEM},
                        {"role": "user",   "content": prompt}]
                fmt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
                out = _reranker_llm.generate([fmt], _sampling)
                raw = out[0].outputs[0].text.strip()
                return self._parse(raw, candidates)

            @staticmethod
            def _parse(text, candidates):
                import json as _j
                from json_repair import repair_json
                try:
                    parsed = _j.loads(text)
                except Exception:
                    try: parsed = _j.loads(repair_json(text))
                    except Exception: parsed = []
                if not isinstance(parsed, list):
                    parsed = []
                fid_set = {c["fragrance_id"] for c in candidates}
                result = [r for r in parsed if r.get("fragrance_id") in fid_set]
                if not result:
                    result = [{"fragrance_id": c["fragrance_id"],
                               "score": c.get("baseline_score", 0.5),
                               "reasoning": "fallback"} for c in candidates]
                return sorted(result, key=lambda x: float(x.get("score",0)), reverse=True)

        reranker = QwenVLReranker()
        RERANKER_BACKEND = "qwen_local"
        print("  Qwen3-VL-32B-Instruct ready (AWQ 4-bit)")
    except Exception as e:
        print(f"  Qwen3-VL-32B load failed: {e}")
        USE_LOCAL_LLM = False

if not USE_LOCAL_LLM:
    bar.set_postfix_str("loading Gemini reranker (API fallback)...")
    from vibescents.reranker import GeminiReranker
    reranker = GeminiReranker()
    RERANKER_BACKEND = "gemini_api"
    print("  Using Gemini API reranker")

bar.update(1)
bar.close()

if os.path.exists("/content"):
    check_disk_space("/content")
print(f"Reranker backend: {RERANKER_BACKEND} | {time.perf_counter()-_t0:.0f}s")


In [ ]:
# Stage 8: Reranker Eval -- rerank top-10 for each of 20 benchmark cases
PIPELINE_VERSION = globals().get("PIPELINE_VERSION", "w3v1")
import time, json, os
import numpy as np
from tqdm.auto import tqdm
from vibescents.similarity import top_k_indices
from vibescents.schemas import RetrievalCandidate
_t0 = time.perf_counter()

RERANK_CACHE = f"{W3_ARTIFACTS}/rerank_results.json"

if os.path.exists(RERANK_CACHE):
    with open(RERANK_CACHE) as fh:
        rerank_results_per_case = json.load(fh)
    print(f"Loaded cached rerank results ({len(rerank_results_per_case)} cases)")
else:
    def _candidate_dicts(top_indices, df, scores):
        cands = []
        for rank, idx in enumerate(top_indices):
            row = df.iloc[int(idx)]
            cands.append({
                "fragrance_id":   str(row.get("fragrance_id", idx)),
                "name":           str(row.get("name", "")),
                "brand":          str(row.get("brand", "")),
                "retrieval_text": str(row.get("retrieval_text", ""))[:300],
                "formality":      float(row.get("formality", 0.5) or 0.5),
                "likely_season":  str(row.get("likely_season", "all-season") or "all-season"),
                "day_night":      float(row.get("day_night", 0.5) or 0.5),
                "baseline_score": float(scores[int(idx)]),
                "rank": rank + 1,
            })
        return cands

    def _gemini_rerank(case, candidate_dicts):
        from vibescents.schemas import RetrievalCandidate
        rc_list = [RetrievalCandidate(
            fragrance_id=c["fragrance_id"], name=c.get("name"),
            brand=c.get("brand"), retrieval_text=c["retrieval_text"],
            baseline_score=c.get("baseline_score"),
            metadata={"formality": c.get("formality"),
                      "likely_season": c.get("likely_season"),
                      "day_night": c.get("day_night")},
        ) for c in candidate_dicts]
        resp = reranker.rerank(occasion_text=case.occasion_text,
                               candidates=rc_list, image_path=None)
        return [{"fragrance_id": r.fragrance_id, "score": r.overall_score,
                 "reasoning": r.explanation} for r in resp.results]

    rerank_results_per_case = []
    pbar = tqdm(enumerate(zip(baseline_results, benchmark_cases)),
                total=N_CASES, desc="Reranking cases", ncols=100)
    for ci, (bres, case) in pbar:
        pbar.set_postfix_str(case.case_id[:30])
        top10_idx  = bres["top10_idx"]
        fused_sc   = bres["fused_scores"]
        cands      = _candidate_dicts(top10_idx, df_enriched, fused_sc)
        try:
            if RERANKER_BACKEND == "qwen_local":
                ranked = reranker.rerank(
                    occasion_text=case.occasion_text,
                    candidates=cands, image_path=None)
            else:
                ranked = _gemini_rerank(case, cands)
        except Exception as exc:
            print(f"\n  [WARN] Reranker failed on {case.case_id}: {exc}")
            ranked = [{"fragrance_id": c["fragrance_id"],
                       "score": c["baseline_score"], "reasoning": "fallback"}
                      for c in cands]
        ranked = sorted(ranked, key=lambda x: float(x.get("score", 0)), reverse=True)
        rerank_results_per_case.append({"case_id": case.case_id, "reranked": ranked})
    pbar.close()

    with open(RERANK_CACHE, "w") as fh:
        json.dump(rerank_results_per_case, fh, indent=2)
    print(f"Saved rerank results to {RERANK_CACHE}")

print(f"Reranking done for {len(rerank_results_per_case)} cases in {time.perf_counter()-_t0:.0f}s")


In [ ]:
# Stage 9: Metric Suite (am@5, am@10, nDCG@10 with delta vs baseline)
PIPELINE_VERSION = globals().get("PIPELINE_VERSION", "w3v1")
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
_t0 = time.perf_counter()

fid_to_idx = {str(df_enriched.iloc[i]["fragrance_id"]): i
              for i in range(len(df_enriched))}

def _am_from_ranked(ranked, ci, k):
    top_k = ranked[:k]
    hits = 0
    for item in top_k:
        fid = item.get("fragrance_id")
        idx = fid_to_idx.get(str(fid))
        if idx is not None:
            hits += int(binary_relevance[ci, idx] > 0)
    return hits / k if k > 0 else 0.0

def _ndcg_from_ranked(ranked, ci, k=10):
    dcg = idcg = 0.0
    for r, item in enumerate(ranked[:k], start=1):
        fid = item.get("fragrance_id")
        idx = fid_to_idx.get(str(fid))
        rel = float(binary_relevance[ci, idx]) if idx is not None else 0.0
        dcg  += rel / np.log2(r + 1)
        idcg += 1.0 / np.log2(r + 1)
    return float(dcg / idcg) if idcg > 0 else 0.0

metrics_rows = []
total_wins   = 0

pbar = tqdm(enumerate(zip(baseline_results, rerank_results_per_case, benchmark_cases)),
            total=N_CASES, desc="Computing metrics", ncols=100)
for ci, (bres, rres, case) in pbar:
    ranked = rres["reranked"]
    r_am5  = _am_from_ranked(ranked, ci, 5)
    r_am10 = _am_from_ranked(ranked, ci, 10)
    r_ndcg = _ndcg_from_ranked(ranked, ci, 10)

    b_am10 = bres["am@10_baseline"]
    b_ndcg = bres["ndcg@10_baseline"]
    win    = int(r_am10 > b_am10)
    total_wins += win

    metrics_rows.append({
        "case_id":           case.case_id,
        "occasion":          case.occasion_text[:50],
        "baseline_am@5":     bres["am@5_baseline"],
        "baseline_am@10":    b_am10,
        "baseline_ndcg@10":  b_ndcg,
        "rerank_am@5":       r_am5,
        "rerank_am@10":      r_am10,
        "rerank_ndcg@10":    r_ndcg,
        "delta_am@10":       r_am10 - b_am10,
        "win":               bool(win),
    })
    pbar.set_postfix(wins=total_wins, am10=f"{r_am10:.3f}")
pbar.close()

metrics_df = pd.DataFrame(metrics_rows)

mean_baseline_am10  = float(metrics_df["baseline_am@10"].mean())
mean_rerank_am10    = float(metrics_df["rerank_am@10"].mean())
mean_delta_am10     = float(metrics_df["delta_am@10"].mean())
mean_baseline_ndcg  = float(metrics_df["baseline_ndcg@10"].mean())
mean_rerank_ndcg    = float(metrics_df["rerank_ndcg@10"].mean())

print("\n" + "=" * 60)
print("AGGREGATE METRICS")
print("=" * 60)
print(f"  mean am@10  baseline: {mean_baseline_am10:.4f}  rerank: {mean_rerank_am10:.4f}  delta: {mean_delta_am10:+.4f}")
print(f"  mean nDCG@10 baseline: {mean_baseline_ndcg:.4f}  rerank: {mean_rerank_ndcg:.4f}")
print(f"  wins: {total_wins}/20")
print(metrics_df[["case_id", "baseline_am@10", "rerank_am@10", "delta_am@10", "win"]].to_string(index=False))
print(f"\nStage 9 done in {time.perf_counter()-_t0:.1f}s")


In [ ]:
# Stage 10: Pre-registered Threshold Check (Decision S)
PIPELINE_VERSION = globals().get("PIPELINE_VERSION", "w3v1")
import json, os

AM10_DELTA_THRESHOLD = 0.03
WINS_THRESHOLD       = 15
NDCG_DELTA_THRESHOLD = 0.0

criterion_am10  = mean_delta_am10  >= AM10_DELTA_THRESHOLD
criterion_wins  = total_wins       >= WINS_THRESHOLD
criterion_ndcg  = mean_rerank_ndcg >= mean_baseline_ndcg + NDCG_DELTA_THRESHOLD

SHIP_RERANKER = criterion_am10 and criterion_wins and criterion_ndcg

print("=" * 60)
print("WEEK 3 RERANKER SHIP GATE -- Decision S")
print("=" * 60)
print(f"  [{'PASS' if criterion_am10 else 'FAIL'}] am@10 delta {mean_delta_am10*100:+.2f}pp  (>= +{AM10_DELTA_THRESHOLD*100:.0f}pp)")
print(f"  [{'PASS' if criterion_wins else 'FAIL'}] wins {total_wins}/20  (>= {WINS_THRESHOLD})")
print(f"  [{'PASS' if criterion_ndcg else 'FAIL'}] nDCG@10 {mean_rerank_ndcg:.4f} vs {mean_baseline_ndcg:.4f}  (no regression)")
print()
if SHIP_RERANKER:
    print("  >>> SHIP: Reranker advances to Week 4. <<<")
else:
    failing = []
    if not criterion_am10: failing.append(f"am@10 delta {mean_delta_am10*100:+.2f}pp < +{AM10_DELTA_THRESHOLD*100:.0f}pp")
    if not criterion_wins:  failing.append(f"wins {total_wins}/20 < {WINS_THRESHOLD}")
    if not criterion_ndcg:  failing.append(f"nDCG regression {mean_rerank_ndcg:.4f} < {mean_baseline_ndcg:.4f}")
    print("  >>> NO-SHIP: Reranker does NOT advance. <<<")
    print(f"  Failing: {'; '.join(failing)}")
    print()
    print("  Week 4 will use 4-signal fusion with LLM-generated reasoning only (no reranker).")
print("=" * 60)

decision = {"ship": SHIP_RERANKER, "am10_delta": mean_delta_am10,
            "wins": total_wins, "ndcg_delta": float(mean_rerank_ndcg - mean_baseline_ndcg),
            "reranker_backend": RERANKER_BACKEND}
DECISION_PATH = f"{W3_ARTIFACTS}/reranker_decision.json"
os.makedirs(os.path.dirname(DECISION_PATH), exist_ok=True)
with open(DECISION_PATH, "w") as fh:
    json.dump(decision, fh, indent=2)
print(f"\nDecision saved to {DECISION_PATH}")


In [ ]:
# Stage 11: Report Writer
PIPELINE_VERSION = globals().get("PIPELINE_VERSION", "w3v1")
import datetime, os, json
from vibescents.fusion import DEFAULT_FUSION_WEIGHTS

REPORT_PATH = f"{REPO_DIR}/results/week3_report.md"

def _table(mdf):
    lines = [
        "| case_id | occasion | base am@10 | rerank am@10 | delta | nDCG base | nDCG rerank | win |",
        "|---------|----------|-----------|-------------|-------|----------|------------|-----|",
    ]
    for _, r in mdf.iterrows():
        lines.append(
            f"| {r['case_id']} | {r['occasion'][:35]} | {r['baseline_am@10']:.3f} "
            f"| {r['rerank_am@10']:.3f} | {r['delta_am@10']:+.3f} "
            f"| {r['baseline_ndcg@10']:.3f} | {r['rerank_ndcg@10']:.3f} "
            f"| {'yes' if r['win'] else 'no'} |"
        )
    return "\n".join(lines)

report = "\n".join([
    "# VibeScent Week 3 Report",
    "",
    f"**Generated:** {datetime.datetime.utcnow().isoformat()} UTC",
    f"**Pipeline version:** {PIPELINE_VERSION}",
    f"**GPU tier:** {GPU_TIER}",
    f"**Reranker backend:** {RERANKER_BACKEND}",
    "",
    "## Enrichment A/B Gate (Decision V)",
    "",
    f"- Gemini mean quality: {gemini_mean:.3f}",
    f"- Qwen mean quality:   {qwen_mean:.3f}",
    f"- Delta:               {qwen_mean - gemini_mean:+.3f}",
    f"- Downstream provider: **{ENRICH_PROVIDER.upper()}**",
    "",
    "## Fusion Weights",
    "",
    f"**Spec (Decision I/Q):** " + "  ".join(f"{k}={v}" for k, v in DEFAULT_FUSION_WEIGHTS.items()),
    "",
    f"**Grid search optimum:** " + "  ".join(f"{k}={v:.2f}" for k, v in OPTIMUM_WEIGHTS.items()),
    "",
    "## Benchmark Results (20 cases)",
    "",
    _table(metrics_df),
    "",
    "## Aggregate Metrics",
    "",
    "| metric | baseline | rerank | delta |",
    "|--------|----------|--------|-------|",
    f"| mean am@10 | {mean_baseline_am10:.4f} | {mean_rerank_am10:.4f} | {mean_delta_am10:+.4f} |",
    f"| mean nDCG@10 | {mean_baseline_ndcg:.4f} | {mean_rerank_ndcg:.4f} | {mean_rerank_ndcg-mean_baseline_ndcg:+.4f} |",
    f"| wins | -- | {total_wins}/20 | -- |",
    "",
    "## Ship Decision (Decision S)",
    "",
    "| criterion | threshold | actual | result |",
    "|-----------|-----------|--------|--------|",
    f"| am@10 delta | >= +3pp | {mean_delta_am10*100:+.2f}pp | {'PASS' if criterion_am10 else 'FAIL'} |",
    f"| wins | >= 15/20 | {total_wins}/20 | {'PASS' if criterion_wins else 'FAIL'} |",
    f"| nDCG@10 | no regression | {mean_rerank_ndcg-mean_baseline_ndcg:+.4f} | {'PASS' if criterion_ndcg else 'FAIL'} |",
    "",
    f"**Overall: {'SHIP -- reranker advances to Week 4' if SHIP_RERANKER else 'NO-SHIP -- Week 4 uses 4-signal fusion only'}**",
])

os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)
with open(REPORT_PATH, "w", encoding="utf-8") as rf:
    rf.write(report)
print(f"Report saved: {REPORT_PATH}")
print(report[:500])


In [ ]:
# Stage 12: Artifact Sink (parallel Drive copies)
PIPELINE_VERSION = globals().get("PIPELINE_VERSION", "w3v1")
import shutil, os, json, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from vibescents.week2_pipeline import write_manifest
import subprocess
_t0 = time.perf_counter()

# ── Save local artifacts ──────────────────────────────────────────────────────
os.makedirs("artifacts/week3", exist_ok=True)
METRICS_CSV   = "artifacts/week3/metrics.csv"
GRID_JSON     = "artifacts/week3/grid_search_results.json"
AB_CSV        = "artifacts/week3/ab_gate_comparison.csv"
MANIFEST_PATH = "artifacts/week3/manifest.json"

metrics_df.to_csv(METRICS_CSV, index=False)
ab_comparison.to_csv(AB_CSV, index=False)

# Grid results (already saved to Drive in Stage 6)
_git = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                      capture_output=True, text=True)
_sha = _git.stdout.strip() if _git.returncode == 0 else "unknown"
write_manifest(MANIFEST_PATH, model="CNNCLIPHybrid+Qwen3-VL-32B",
               commit_sha=_sha, row_count=len(metrics_df), dim=0,
               pipeline_version=PIPELINE_VERSION)

# ── Parallel Drive copy ───────────────────────────────────────────────────────
copy_tasks = []
for src in [METRICS_CSV, GRID_JSON, AB_CSV, MANIFEST_PATH,
            f"{REPO_DIR}/results/week3_report.md",
            f"{W3_ARTIFACTS}/reranker_decision.json"]:
    if os.path.exists(src):
        dst = f"{W3_ARTIFACTS}/{os.path.basename(src)}"
        copy_tasks.append((src, dst))

pbar = tqdm(total=len(copy_tasks), desc="Drive sync", ncols=90, unit="file")
with ThreadPoolExecutor(max_workers=4) as pool:
    futs = {pool.submit(shutil.copy2, s, d): os.path.basename(s)
            for s, d in copy_tasks}
    for fut in as_completed(futs):
        try:
            fut.result()
            pbar.set_postfix_str(futs[fut])
        except Exception as e:
            print(f"\n  [WARN] Copy failed: {e}")
        pbar.update(1)
pbar.close()

print("\n" + "-" * 55)
print("Git-add snippet:")
print("-" * 55)
print("git add artifacts/week3/ results/week3_report.md")
print(f'git commit -m "feat: Week 3 -- ship={SHIP_RERANKER}, '
      f'am10_delta={mean_delta_am10*100:+.2f}pp, wins={total_wins}/20"')
print("-" * 55)
print(f"\nAll done in {time.perf_counter()-_t0:.1f}s  |  Ship: {SHIP_RERANKER}")
